In [ ]:
# 1. INSTALLATION
!pip install catboost==1.2.8 scikit-learn==1.6.1 ta==0.11.0 pandas_datareader==0.10.0 tabulate requests matplotlib > /dev/null 2>&1

# 2. DRIVE MOUNT & PATH CONFIG
import os
import sys
from google.colab import drive

if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

BASE_PATH = '/content/drive/MyDrive/Global_Macro_Model'
os.makedirs(BASE_PATH, exist_ok=True)
os.chdir(BASE_PATH)

# --- ARTIFACTS FOLDER ---
ARTIFACTS_DIR = 'artifacts' 
os.makedirs(ARTIFACTS_DIR, exist_ok=True)
os.makedirs('data_cache', exist_ok=True)

# 3. CORE LOGIC
import time
import requests
import warnings
import joblib
import logging
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from pandas_datareader import data as pdr
from ta.momentum import RSIIndicator
from ta.trend import MACD
from ta.volatility import BollingerBands, AverageTrueRange
from catboost import CatBoostClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.calibration import CalibratedClassifierCV
from tabulate import tabulate

try:
    from zoneinfo import ZoneInfo
except ImportError:
    from backports.zoneinfo import ZoneInfo

# --- SILENCE WARNINGS ---
os.environ['PYTHONWARNINGS'] = 'ignore'
warnings.filterwarnings('ignore')
warnings.filterwarnings("ignore", message=".*cv='prefit'.*")
logging.getLogger('yfinance').setLevel(logging.CRITICAL)
logging.getLogger('numexpr').setLevel(logging.CRITICAL)

# --- CONFIGURATION (SPY ONLY) ---
INITIAL_CAPITAL = 2500
FAST_MODE = True 

ASSET_CONFIG = {
    'SPY': {
        'target_pct': 0.0025,
        'win_amount': 55,
        'loss_amount': 45,
        'description': 'S&P 500',
    }
}

TWELVEDATA_API_KEY = os.getenv("TWELVEDATA_API_KEY", "7c5a55559d164d4ebca0d23a5baea839").strip()

# --- DATA ENGINE ---
def download_fred_series(series, start):
    try:
        df = pdr.DataReader(series, "fred", start)
        return df.dropna()
    except: return pd.DataFrame()

def _fred_to_ohlcv(series_df: pd.DataFrame, col: str) -> pd.DataFrame:
    if series_df.empty: return pd.DataFrame()
    s = series_df[col]
    df = pd.DataFrame(index=s.index)
    df['Close'] = s; df['Open'] = s; df['High'] = s; df['Low'] = s; df['Volume'] = 0
    if df.index.tz is not None: df.index = df.index.tz_localize(None)
    return df

def download_twelvedata_robust(symbol, api_key, limit=5000):
    if not api_key: return pd.DataFrame()
    url = "https://api.twelvedata.com/time_series"
    params = {'symbol': symbol, 'interval': '1day', 'outputsize': limit, 'format': 'JSON', 'apikey': api_key, 'order': 'ASC'}
    try:
        for attempt in range(3):
            r = requests.get(url, params=params, timeout=45)
            if r.status_code == 200: break
            time.sleep(2)
        if r.status_code != 200: return pd.DataFrame()
        j = r.json()
        if 'values' not in j: return pd.DataFrame()
        df = pd.DataFrame(j['values'])
        df['datetime'] = pd.to_datetime(df['datetime'])
        df = df.set_index('datetime').sort_index()
        rename = {'open': 'Open', 'high': 'High', 'low': 'Low', 'close': 'Close', 'volume': 'Volume'}
        df = df.rename(columns=rename)
        for col in ['Open', 'High', 'Low', 'Close']: df[col] = pd.to_numeric(df[col], errors='coerce')
        df['Volume'] = pd.to_numeric(df.get('Volume', 0), errors='coerce').fillna(0)
        if df.index.tz is not None: df.index = df.index.tz_localize(None)
        return df[['Open', 'High', 'Low', 'Close', 'Volume']]
    except: return pd.DataFrame()

def prepare_data(ticker, config, is_inference=False):
    # --- FIX: ALWAYS USE 5000 CANDLES TO PREVENT INDICATOR DRIFT ---
    limit = 5000 
    
    print(f"Downloading data for {ticker} (Limit={limit})...", end=" ", flush=True)
    
    # 1. Main Asset
    asset = download_twelvedata_robust(ticker, TWELVEDATA_API_KEY, limit=limit)
    
    # Validation for inference safety
    min_required = 200 if is_inference else 3000
    if len(asset) < min_required:
        print("FAILED.")
        raise ValueError(f"CRITICAL: Insufficient data for {ticker}. Got {len(asset)} rows.")

    start_dt = asset.index.min()
    
    # 2. FRED Data
    vix = download_fred_series("VIXCLS", start=start_dt)
    vix = _fred_to_ohlcv(vix, 'VIXCLS')
    tnx = download_fred_series("DGS10", start=start_dt)
    tnx = _fred_to_ohlcv(tnx, 'DGS10')
    
    # 3. Macro ETFs
    # We maintain sleep to avoid rate limits, but for inference we can be faster if needed
    if not is_inference: time.sleep(4)
    hyg = download_twelvedata_robust("HYG", TWELVEDATA_API_KEY, limit=limit)
    if not is_inference: time.sleep(4)
    xlk = download_twelvedata_robust("XLK", TWELVEDATA_API_KEY, limit=limit)
    if not is_inference: time.sleep(4)
    uup = download_twelvedata_robust("UUP", TWELVEDATA_API_KEY, limit=limit)
    if not is_inference: time.sleep(4)
    uso = download_twelvedata_robust("USO", TWELVEDATA_API_KEY, limit=limit)

    # 4. Align Data
    closes = pd.DataFrame(index=asset.index)
    closes[ticker] = asset["Close"]
    
    # Robust reindexing
    closes["^VIX"] = vix["Close"].reindex(closes.index).ffill().bfill()
    closes["^TNX"] = tnx["Close"].reindex(closes.index).ffill().bfill()
    closes["HYG"] = hyg["Close"].reindex(closes.index).ffill().bfill()
    closes["XLK"] = xlk["Close"].reindex(closes.index).ffill().bfill()
    closes["UUP"] = uup["Close"].reindex(closes.index).ffill().bfill()
    closes["USO"] = uso["Close"].reindex(closes.index).ffill().bfill()
    
    asset_open = asset["Open"]; asset_close = asset["Close"]
    asset_high = asset["High"]; asset_low = asset["Low"]
    asset_volume = asset["Volume"]
    vix_open = vix["Open"].reindex(closes.index).ffill().bfill()
    vix_close = vix["Close"].reindex(closes.index).ffill().bfill()
    
    _df = closes.copy()
    
    # 5. Log Returns
    base_tickers = [ticker, '^VIX', '^TNX', 'HYG', 'XLK', 'UUP', 'USO']
    for t in base_tickers:
        if t in _df.columns: 
            _df[f'{t}_Log_Return_Close'] = np.log(_df[t] / _df[t].shift(1)).shift(1)
            _df[f'{t}_3D_Mom'] = (_df[t].shift(1) - _df[t].shift(4)) / _df[t].shift(4)

    # 6. Gaps
    _df[f'{ticker}_Overnight_Gap'] = (asset_open - asset_close.shift(1)) / asset_close.shift(1)
    _df['VIX_Overnight_Gap'] = (vix_open - vix_close.shift(1)) / vix_close.shift(1)
    
    # 7. Lags
    lags = [1, 2, 3, 5]
    for lag in lags:
        _df[f'{ticker}_Return_Lag{lag}'] = _df[f'{ticker}_Log_Return_Close'].shift(lag)
        _df[f'VIX_Return_Lag{lag}'] = _df['^VIX_Log_Return_Close'].shift(lag)
    
    # 8. Volatility
    for w in [5, 10, 20]: 
        _df[f'{ticker}_Roll_Std_{w}'] = _df[f'{ticker}_Log_Return_Close'].rolling(window=w).std()

    # 9. Tech Indicators
    close_s = _df[ticker]
    sma20 = close_s.rolling(20).mean().shift(1)
    sma50 = close_s.rolling(50).mean().shift(1)
    sma200 = close_s.rolling(200).mean().shift(1)
    
    _df[f'{ticker}_SMA20_Dist'] = (close_s.shift(1) / sma20) - 1
    _df[f'{ticker}_SMA200_Dist'] = (close_s.shift(1) / sma200) - 1
    
    rsi = RSIIndicator(close=close_s, window=14).rsi().shift(1)
    _df[f'{ticker}_RSI'] = rsi
    _df[f'{ticker}_RSI_Slope'] = rsi - rsi.shift(3) 

    macd_obj = MACD(close=close_s, window_slow=26, window_fast=12, window_sign=9)
    _df['MACD'] = macd_obj.macd().shift(1)
    _df['MACD_Signal'] = macd_obj.macd_signal().shift(1)
    _df['MACD_Hist'] = _df['MACD'] - _df['MACD_Signal']

    bb = BollingerBands(close=close_s, window=20, window_dev=2)
    _df[f'{ticker}_BB_Width'] = ((bb.bollinger_hband() - bb.bollinger_lband()) / bb.bollinger_mavg()).shift(1)
    
    atr = AverageTrueRange(high=asset_high, low=asset_low, close=asset_close, window=14).average_true_range()
    _df[f'{ticker}_ATR'] = atr.shift(1)
    
    price_change = asset_close.diff()
    direction = np.sign(price_change).fillna(0)
    obv = (direction * asset_volume).cumsum()
    _df[f'{ticker}_OBV'] = obv.shift(1)
    
    _df['Day_of_Week'] = _df.index.dayofweek
    _df['Day_of_Month'] = _df.index.day
    
    # 10. Target Generation
    target_pct = config['target_pct']
    intraday_change = (asset_close - asset_open) / asset_open
    conditions = [(intraday_change > target_pct), (intraday_change < -target_pct)]
    choices = [1, -1]
    _df['Target'] = np.select(conditions, choices, default=0)
    
    _df.dropna(inplace=True)
    _df.index = _df.index.tz_localize(None)
    print("Done.")
    return _df

# --- MODEL UTILS ---
def apply_rule_from_probs(probs, threshold, max_noise, margin):
    bear_p = probs[:, 0]; noise_p = probs[:, 1]; bull_p = probs[:, 2]
    pred = np.ones(len(probs), dtype=int)
    bear_mask = (bear_p > threshold)
    bull_mask = (bull_p > threshold)
    if max_noise < 1.0:
        bear_mask = bear_mask & (noise_p <= max_noise)
        bull_mask = bull_mask & (noise_p <= max_noise)
    pred[bear_mask] = 0; pred[bull_mask] = 2
    return pred

def make_base_model(fast):
    return CatBoostClassifier(
        loss_function='MultiClass', iterations=600 if fast else 1500, 
        depth=6, learning_rate=0.05, l2_leaf_reg=6, 
        random_seed=42, allow_writing_files=False, logging_level='Silent'
    )

def fit_model_full(X, y_mapped, fast):
    n = len(X); n_cal = max(60, int(n * 0.15))
    X_fit, y_fit = X[:-n_cal], y_mapped[:-n_cal]
    X_cal, y_cal = X[-n_cal:], y_mapped[-n_cal:]
    base = make_base_model(fast)
    base.fit(X_fit, y_fit)
    cal = CalibratedClassifierCV(base, method='sigmoid', cv='prefit')
    cal.fit(X_cal, y_cal)
    return cal

def oof_probs_tabular(X, y_mapped, fast):
    n_splits = 3 if fast else 5
    tscv = TimeSeriesSplit(n_splits=n_splits)
    oof = np.full((len(X), 3), np.nan, dtype=float)
    for tr_idx, val_idx in tscv.split(X):
        model = fit_model_full(X[tr_idx], y_mapped[tr_idx], fast=fast)
        oof[val_idx] = model.predict_proba(X[val_idx])
    mask = ~np.isnan(oof).any(axis=1)
    return oof[mask], mask

def optimize_rule_on_train(probs_train, true_target_train, win_amount, loss_amount, min_trades=20):
    threshold_grid = np.arange(0.35, 0.85, 0.02)
    max_noise_grid = [1.00, 0.80, 0.60]
    best = {'threshold': 0.60, 'max_noise': 1.00, 'margin': 0.00, 'ev_total': -1e9}
    
    for max_noise in max_noise_grid:
        for th in threshold_grid:
            pred = apply_rule_from_probs(probs_train, th, max_noise, margin=0.0)
            trade_mask = pred != 1
            if np.sum(trade_mask) < min_trades: continue
            
            p_dir = np.where(pred[trade_mask] == 0, -1, 1)
            actual = true_target_train[trade_mask]
            wins = np.sum(p_dir == actual)
            losses = len(p_dir) - wins
            ev = wins * win_amount - losses * loss_amount
            if ev > best['ev_total']:
                 best = {'threshold': float(th), 'max_noise': float(max_noise), 'margin': 0.0, 'ev_total': ev}
    return best

# ------------------------------------------------------------
# WEEKEND TRAINING
# ------------------------------------------------------------
def run_weekend_training_mode():
    print(f"\n{'='*70}")
    print(f"  WEEKEND TRAINING & HEALTH CHECK (SPY ONLY) - MACRO VERSION")
    print(f"  ARTIFACTS WILL BE SAVED TO: {ARTIFACTS_DIR}")
    print(f"{'='*70}")
    
    for ticker, config in ASSET_CONFIG.items():
        try:
            # 1. DATA
            df = prepare_data(ticker, config, is_inference=False)
            
            exclude = ['Target', ticker, '^VIX', '^TNX', 'HYG', 'XLK', 'UUP', 'USO']
            feats = [c for c in df.columns if c not in exclude]
            X_all = df[feats].values
            y_all = (df['Target'].values + 1).astype(int)
            y_targets = df['Target'].values
            dates = df.index
            
            history_log = []
            
            # --- PHASE 1: LONG-TERM STABILITY ---
            print(f"\n--- {ticker} PHASE 1: LONG-TERM STABILITY ---")
            
            sim_start_date = pd.Timestamp("2015-01-01")
            start_idx = df.index.searchsorted(sim_start_date)
            if start_idx < 500: start_idx = 500
            
            current_idx = start_idx
            cutoff_1year = df.index.searchsorted(pd.Timestamp.now() - pd.Timedelta(days=365))
            
            while current_idx < cutoff_1year:
                end_idx = min(current_idx + 63, cutoff_1year)
                if end_idx == current_idx: break
                
                X_tr = X_all[:current_idx]; y_tr = y_all[:current_idx]
                X_te = X_all[current_idx:end_idx]; y_te_targ = y_targets[current_idx:end_idx]
                
                oof, mask = oof_probs_tabular(X_tr, y_tr, fast=FAST_MODE)
                rule = optimize_rule_on_train(oof, y_targets[:current_idx][mask], config['win_amount'], config['loss_amount'])
                model = fit_model_full(X_tr, y_tr, fast=FAST_MODE)
                
                probs = model.predict_proba(X_te)
                pred = apply_rule_from_probs(probs, rule['threshold'], rule['max_noise'], rule['margin'])
                
                batch_dates = dates[current_idx:end_idx]
                for i in range(len(pred)):
                    p_val = pred[i]
                    actual = y_te_targ[i]
                    outcome_r = 0
                    if p_val != 1:
                        p_dir = 1 if p_val == 2 else -1
                        outcome_r = 1 if p_dir == actual else -1
                    history_log.append({'date': batch_dates[i], 'outcome_r': outcome_r})
                current_idx = end_idx

            # --- PHASE 2: LAST 52 WEEKS ---
            print(f"\n--- {ticker} PHASE 2: LAST 52 WEEKS (Micro Review) ---")
            start_search_idx = cutoff_1year
            while start_search_idx < len(df):
                if df.index[start_search_idx].dayofweek == 0: break
                start_search_idx += 1
            
            current_idx = start_search_idx
            total_len = len(X_all)
            
            while current_idx < total_len:
                curr_year, curr_week, _ = df.index[current_idx].isocalendar()
                batch_indices = []
                temp_idx = current_idx
                while temp_idx < total_len:
                    y, w, _ = df.index[temp_idx].isocalendar()
                    if y == curr_year and w == curr_week:
                        batch_indices.append(temp_idx)
                        temp_idx += 1
                    else: break
                
                if not batch_indices: break
                batch_start = batch_indices[0]
                batch_end = batch_indices[-1] + 1
                
                X_tr = X_all[:batch_start]; y_tr = y_all[:batch_start]
                X_te = X_all[batch_start:batch_end]; y_te_targ = y_targets[batch_start:batch_end]
                
                if len(y_tr) < 100: 
                    current_idx = batch_end
                    continue

                oof, mask = oof_probs_tabular(X_tr, y_tr, fast=FAST_MODE)
                rule = optimize_rule_on_train(oof, y_targets[:batch_start][mask], config['win_amount'], config['loss_amount'])
                model = fit_model_full(X_tr, y_tr, fast=FAST_MODE)
                
                probs = model.predict_proba(X_te)
                pred = apply_rule_from_probs(probs, rule['threshold'], rule['max_noise'], rule['margin'])
                
                batch_dates = dates[batch_start:batch_end]
                for i in range(len(pred)):
                    p_val = pred[i]
                    actual = y_te_targ[i]
                    outcome_r = 0
                    if p_val != 1:
                        p_dir = 1 if p_val == 2 else -1
                        outcome_r = 1 if p_dir == actual else -1
                    history_log.append({'date': batch_dates[i], 'outcome_r': outcome_r})

                n_trades = np.sum(pred != 1)
                n_wins = 0
                if n_trades > 0:
                    p_dirs = np.where(pred[pred!=1]==0, -1, 1)
                    acts = y_te_targ[pred!=1]
                    n_wins = np.sum(p_dirs == acts)
                    wr = (n_wins / n_trades) * 100
                else: wr = 0.0
                
                d_str = f"{dates[batch_start].date()} -> {dates[batch_end-1].date()}"
                print(f"  {d_str} | Trades: {n_trades} | WinRate: {wr:5.1f}%")
                
                current_idx = batch_end

            # --- PHASE 3: LAST 10 TRADES PREDICTIONS ---
            print(f"\n--- {ticker} PHASE 3: LAST 10 TRADES PREDICTIONS ---")
            print("  NOTE: This section uses a 'Validation Model' trained on data up to 10 days ago.")
            print("  Discrepancies with 'Live' predictions may occur if Thresholds evolve.")
            
            n_history = 10
            if len(X_all) > n_history + 200:
                X_tr_final = X_all[:-n_history]
                y_tr_final = y_all[:-n_history]
                X_te_final = X_all[-n_history:]
                y_te_final_target = y_targets[-n_history:]
                dates_final = dates[-n_history:]
                
                model_p3 = fit_model_full(X_tr_final, y_tr_final, fast=FAST_MODE)
                oof_p3, mask_p3 = oof_probs_tabular(X_tr_final, y_tr_final, fast=FAST_MODE)
                rule_p3 = optimize_rule_on_train(oof_p3, y_targets[:-n_history][mask_p3], config['win_amount'], config['loss_amount'])
                
                print(f"  Phase 3 (Validation) Threshold: {rule_p3['threshold']:.2f}")

                probs_p3 = model_p3.predict_proba(X_te_final)
                preds_p3 = apply_rule_from_probs(probs_p3, rule_p3['threshold'], rule_p3['max_noise'], rule_p3['margin'])
                
                table_data = []
                for i in range(len(dates_final)):
                    d_date = dates_final[i].date()
                    d_day = dates_final[i].strftime("%A")
                    p_val = preds_p3[i]
                    p_str = "BULL" if p_val == 2 else ("BEAR" if p_val == 0 else "NOISE")
                    
                    bear_prob = probs_p3[i][0] * 100
                    bull_prob = probs_p3[i][2] * 100
                    
                    actual_val = y_te_final_target[i] 
                    act_str = "BULL" if actual_val == 1 else ("BEAR" if actual_val == -1 else "FLAT")
                    
                    outcome = ""
                    if p_val == 1: outcome = "---"
                    else:
                        dir_pred = 1 if p_val == 2 else -1
                        if dir_pred == actual_val: outcome = "WIN"
                        else: outcome = "LOSS"
                    
                    table_data.append([d_date, d_day, f"{bear_prob:.1f}%", f"{bull_prob:.1f}%", p_str, act_str, outcome])
                    
                print(tabulate(table_data, headers=["Date", "Day", "Bear%", "Bull%", "Sim_Pred", "Actual", "Result"], tablefmt="grid"))

            # PHASE 4: FINAL TRAIN & SAVE
            print(f"\n[{ticker}] Training Final Model on Full History (Including last 10 days)...")
            final_model = fit_model_full(X_all, y_all, fast=FAST_MODE)
            oof_probs, mask = oof_probs_tabular(X_all, y_all, fast=FAST_MODE)
            final_rule = optimize_rule_on_train(oof_probs, y_targets[mask], config['win_amount'], config['loss_amount'], min_trades=30)
            
            print(f"[{ticker}] SAVING FINAL PARAMETERS: Threshold={final_rule['threshold']:.2f}")
            
            save_dir = os.path.join(ARTIFACTS_DIR, ticker)
            os.makedirs(save_dir, exist_ok=True)
            joblib.dump(final_model, os.path.join(save_dir, 'tabular_model.pkl'))
            joblib.dump(feats, os.path.join(save_dir, 'feature_cols.pkl'))
            with open(os.path.join(save_dir, 'threshold.txt'), 'w') as f: f.write(str(final_rule['threshold']))
            with open(os.path.join(save_dir, 'max_noise_prob.txt'), 'w') as f: f.write(str(final_rule['max_noise']))
            with open(os.path.join(save_dir, 'margin_vs_noise.txt'), 'w') as f: f.write(str(final_rule['margin']))
            
        except Exception as e:
            print(f"ERROR training {ticker}: {e}")
            import traceback
            traceback.print_exc()

# ------------------------------------------------------------
# 5. INFERENCE FUNCTION (Run Daily)
# ------------------------------------------------------------
def run_inference_mode():
    now_et = datetime.now(ZoneInfo("America/New_York"))
    print(f"\n{'#'*60}")
    print(f"  LIVE INFERENCE - {now_et.strftime('%Y-%m-%d %H:%M')} ET")
    print(f"  LOADING ARTIFACTS FROM: {ARTIFACTS_DIR}")
    print(f"{'#'*60}")
    
    for ticker in ASSET_CONFIG.keys():
        save_dir = os.path.join(ARTIFACTS_DIR, ticker)
        if not os.path.exists(save_dir): continue
            
        try:
            model = joblib.load(os.path.join(save_dir, 'tabular_model.pkl'))
            feats = joblib.load(os.path.join(save_dir, 'feature_cols.pkl'))
            with open(os.path.join(save_dir, 'threshold.txt'), 'r') as f: th = float(f.read().strip())
            with open(os.path.join(save_dir, 'max_noise_prob.txt'), 'r') as f: mn = float(f.read().strip())
            with open(os.path.join(save_dir, 'margin_vs_noise.txt'), 'r') as f: mg = float(f.read().strip())
            
            # --- CRITICAL: Use same data limit as training to ensure feature parity ---
            df = prepare_data(ticker, ASSET_CONFIG[ticker], is_inference=True)
            if len(df) == 0: continue
            
            X = np.zeros((len(df), len(feats)))
            for i, col in enumerate(feats):
                if col in df.columns: X[:, i] = df[col].values
            
            probs = model.predict_proba(X)
            pred = apply_rule_from_probs(probs, th, mn, mg)
            
            latest_idx = -1
            latest_date = df.index[latest_idx]
            latest_probs = probs[latest_idx]
            latest_sig = pred[latest_idx]
            
            sig_str = {0: 'BEAR', 1: 'NOISE', 2: 'BULL'}[latest_sig]
            
            print(f"\n>>> {ticker} SIGNAL ({latest_date.date()}) <<<")
            print(f"Prediction: {sig_str}")
            print(f"Probabilities: Bear={latest_probs[0]:.1%} | Noise={latest_probs[1]:.1%} | Bull={latest_probs[2]:.1%}")
            print(f"Threshold Used: {th}")
            
            if latest_sig == 2: print(f"ACTION: BUY CALL DEBIT SPREAD")
            elif latest_sig == 0: print(f"ACTION: BUY PUT DEBIT SPREAD")
            else: print("ACTION: WAIT / NO TRADE")
                
        except Exception as e:
            print(f"Error inference {ticker}: {e}")

# ============================================================
# EXECUTION CONTROL
# ============================================================

# Uncomment one:
run_weekend_training_mode()
# run_inference_mode()